# Session 9 · Introduction to Apache Pig

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Hamnakhan123/Diploma-in-Data-Science/blob/main/term-2/02-big-data-systems-with-hadoop/code/session-09-introduction-to-apache-pig.ipynb)

Diploma in Data Science · Prepared by **Hamna Zahid Khan**, Junior Lecturer

Run the cells from top to bottom. Every example is self-contained, so you can also run any single cell on its own.

**Setup:** the Spark examples need PySpark (and Java, which Google Colab already has).

In [1]:
%pip install -q pyspark

### Example 1 · Tracing the script step by step (Pig semantics in Python)

In [2]:
from collections import defaultdict
sales = [(1, "Karachi", 2500.0), (2, "Lahore", 800.0), (3, "Karachi", 1200.0),
         (4, "Islamabad", 3100.0), (5, "Lahore", 1500.0), (6, "Karachi", 400.0), (7, "Lahore", 2600.0)]
print("sales   :", len(sales), "tuples")

big = [t for t in sales if t[2] > 1000]                      # FILTER
print("big     :", big)

by_city = defaultdict(list)                                   # GROUP big BY city
for t in big:
    by_city[t[1]].append(t)
print("by_city :")
for city, bag in by_city.items():
    print(f"   ({city}, {{{', '.join('(' + ','.join(map(str, t)) + ')' for t in bag)}}})")

totals = [(c, len(bag), sum(t[2] for t in bag)) for c, bag in by_city.items()]   # FOREACH GENERATE
ranked = sorted(totals, key=lambda r: -r[2])                                      # ORDER BY DESC
print("ranked  :", ranked)

sales   : 7 tuples
big     : [(1, 'Karachi', 2500.0), (3, 'Karachi', 1200.0), (4, 'Islamabad', 3100.0), (5, 'Lahore', 1500.0), (7, 'Lahore', 2600.0)]
by_city :
   (Karachi, {(1,Karachi,2500.0), (3,Karachi,1200.0)})
   (Islamabad, {(4,Islamabad,3100.0)})
   (Lahore, {(5,Lahore,1500.0), (7,Lahore,2600.0)})
ranked  : [('Lahore', 2, 4100.0), ('Karachi', 2, 3700.0), ('Islamabad', 1, 3100.0)]


### Example 2 · The same data flow in PySpark

In [3]:
from pyspark.sql import SparkSession, functions as F
spark = (SparkSession.builder.master("local[2]").appName("pig-style")
         .config("spark.ui.showConsoleProgress", "false").getOrCreate())
spark.sparkContext.setLogLevel("ERROR")

sales = spark.createDataFrame(
    [(1, "Karachi", 2500.0), (2, "Lahore", 800.0), (3, "Karachi", 1200.0), (4, "Islamabad", 3100.0),
     (5, "Lahore", 1500.0), (6, "Karachi", 400.0), (7, "Lahore", 2600.0)],
    "order_id int, city string, amount double")                         # LOAD ... AS (...)
big = sales.filter(F.col("amount") > 1000)                               # FILTER
by_city = big.groupBy("city")                                            # GROUP
totals = by_city.agg(F.count("*").alias("orders"), F.sum("amount").alias("revenue"))  # FOREACH GENERATE
ranked = totals.orderBy(F.desc("revenue"))                               # ORDER
ranked.show()                                                            # DUMP
ranked.explain()                                                         # EXPLAIN

+---------+------+-------+
|     city|orders|revenue|
+---------+------+-------+
|   Lahore|     2| 4100.0|
|  Karachi|     2| 3700.0|
|Islamabad|     1| 3100.0|
+---------+------+-------+

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- Sort [revenue#4 DESC NULLS LAST], true, 0
   +- Exchange rangepartitioning(revenue#4 DESC NULLS LAST, 200), ENSURE_REQUIREMENTS, [plan_id=87]
      +- HashAggregate(keys=[city#1], functions=[count(1), sum(amount#2)])
         +- Exchange hashpartitioning(city#1, 200), ENSURE_REQUIREMENTS, [plan_id=84]
            +- HashAggregate(keys=[city#1], functions=[partial_count(1), partial_sum(amount#2)])
               +- Project [city#1, amount#2]
                  +- Filter (isnotnull(amount#2) AND (amount#2 > 1000.0))
                     +- Scan ExistingRDD[order_id#0,city#1,amount#2]




### Example 3 · Same question, four languages

In [4]:
versions = {"MapReduce (Java)": 60, "Pig Latin": 4, "HiveQL / SQL": 1, "PySpark DataFrame": 1}
for lang, lines in versions.items():
    print(f"{lang:<18} {'#' * lines if lines < 40 else '#' * 40 + ' ...'} {lines}")

MapReduce (Java)   ######################################## ... 60
Pig Latin          #### 4
HiveQL / SQL       # 1
PySpark DataFrame  # 1
